In [ ]:
import os
import random
import numpy as np
from collections import deque

import vizdoom as vzd

import torch
import torch.nn as nn
import torch.optim as optim

def resetGame(visibilityOn):
    # =========================
    # ENV SETUP
    # =========================
    game = vzd.DoomGame()

    config = {
        "doom_scenario_path": os.path.join(vzd.scenarios_path, "basic.wad"),
        "doom_map": "map01",

        "available_buttons": [
            vzd.Button.MOVE_LEFT,
            vzd.Button.MOVE_RIGHT,
            vzd.Button.ATTACK,
        ],

        # MUST enable richer state
        "available_game_variables": [
            vzd.GameVariable.HEALTH,
            vzd.GameVariable.AMMO2,
            vzd.GameVariable.POSITION_X,
            vzd.GameVariable.POSITION_Y
        ],

        "episode_timeout": 300,
        "episode_start_time": 10,
        "living_reward": -0.01,
        "mode": vzd.Mode.PLAYER,
    }

    game.set_config(config)

    # 🔥 IMPORTANT: enable object info
    game.set_objects_info_enabled(True)
    game.set_window_visible(visibilityOn)

    game.init()

    return game


# =========================
# ACTION SPACE
# =========================
ACTIONS = [
    [True, False, False],   # left
    [False, True, False],   # right
    [False, False, True],   # shoot
    [False, False, False],  # noop
]


# =========================
# STATE ENCODING (NO VISION, OBJECT-AWARE)
# =========================
def get_state_vector(state):
    health, ammo, pos_x, pos_y = state.game_variables

    health = health / 100.0
    ammo = ammo / 50.0



    enemy_x = 0.0
    enemy_y = 0.0

    if state.objects is not None and len(state.objects) > 0:
        # print("State:")
        # print(state.objects)
        # for object in state.objects:
        #     print(object.name)
        enemies = [o for o in state.objects if "Cacodemon" in o.name]

        if len(enemies) > 0:
            enemy = enemies[0]

            enemy_x = enemy.position_x
            enemy_y = enemy.position_y

    return np.array([
        health,
        ammo,
        pos_x,
        pos_y,
        enemy_x,
        enemy_y
    ], dtype=np.float32)

# =========================
# Q NETWORK (MLP)
# =========================
class DQN(nn.Module):
    def __init__(self, n_actions):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(6, 64),   # 🔥 now 3 inputs
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, n_actions)
        )

    def forward(self, x):
        return self.net(x)


# =========================
# TRAIN SETUP
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DQN(len(ACTIONS)).to(device)
target_model = DQN(len(ACTIONS)).to(device)
target_model.load_state_dict(model.state_dict())

optimizer = optim.Adam(model.parameters(), lr=1e-3)

memory = deque(maxlen=5000)

gamma = 0.99
epsilon = 1.0
epsilon_min = 0.1
epsilon_decay = 0.995

batch_size = 32
target_update = 10


# =========================
# ACTION SELECT
# =========================
def select_action(state_vec):
    global epsilon

    if random.random() < epsilon:
        return random.randint(0, len(ACTIONS) - 1)

    with torch.no_grad():
        q = model(state_vec)
        return torch.argmax(q).item()


# =========================
# TRAIN LOOP
# =========================
episodes = 300

game = resetGame(False)

for episode in range(episodes + 1):
    if(episode % 5 == 0):
        game.close()
        game = resetGame(True)
    
    game.new_episode()

    state = game.get_state()
    state_vec = torch.tensor(get_state_vector(state)).unsqueeze(0).to(device)

    total_reward = 0

    while not game.is_episode_finished():

        action = select_action(state_vec)
        reward = game.make_action(ACTIONS[action])

        total_reward += reward

        if not game.is_episode_finished():
            next_state = game.get_state()
            next_vec = torch.tensor(get_state_vector(next_state)).unsqueeze(0).to(device)
        else:
            next_vec = None

        memory.append((state_vec, action, reward, next_vec))

        state_vec = next_vec if next_vec is not None else state_vec

        # =========================
        # TRAIN STEP
        # =========================
        if len(memory) > batch_size:

            batch = random.sample(memory, batch_size)
            s, a, r, s2 = zip(*batch)

            s = torch.cat(s)
            a = torch.tensor(a).to(device)
            r = torch.tensor(r).float().to(device)

            q = model(s).gather(1, a.unsqueeze(1)).squeeze()

            target = q.clone().detach()

            for i in range(batch_size):
                if s2[i] is not None:
                    target[i] = r[i] + gamma * target_model(s2[i]).max()
                else:
                    target[i] = r[i]

            loss = ((q - target) ** 2).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if episode % target_update == 0:
        target_model.load_state_dict(model.state_dict())

    print(f"Episode {episode} | Reward: {total_reward:.2f} | Epsilon: {epsilon:.3f}")

    if(episode % 5 == 0):
        game.close()
        game = resetGame(False)

game.close()